# MIMIC-IV temporal validation of explanation quality

Temporal validation of model performance and explanation quality on MIMIC-IV.


Development era: anchor_year_group 2008-2016. Temporal validation era: 2017-2022.
The identical feature definitions (extract_mimic.FEATURE_MAP) are used in both
eras. Steps:
  1. feature-mapping check: missingness, medians and standardised mean
     differences (SMD) per feature in each era
  2. models tuned and trained on the development era (70%); evaluated on the
     development hold-out (30%, internal) and on the whole 2017-2022 era (temporal)
  3. explanation quality (deletion faithfulness, perturbation stability,
     explainer agreement, complexity) computed on 100 internal and 100 temporal
     instances with the same explainers as the open-data benchmark
  4. explanation transfer: global importance ranking internal vs temporal, and
     local agreement between the development model and a model refitted on the
     2017-2022 era (same specification), on the same temporal instances
All tables are written as CSV and as LaTeX into <out>/, and the headline numbers
as LaTeX macros (<out>/mimic_macros.tex) that the manuscript reads.

## Setup
Loads `00_config` (paths) and the library notebooks from this folder.

In [11]:
# Load other notebooks in this folder as modules (all notebooks live in one folder).
import os, sys, types
import nbformat

NB_DIR = os.getcwd()   # open Jupyter in the notebooks folder


def import_notebook(name, filename):
    """Execute the code cells of a notebook in this folder into a module named `name`."""
    if name in sys.modules:
        return sys.modules[name]
    path = os.path.join(NB_DIR, filename)
    mod = types.ModuleType(name)
    mod.__file__ = path
    sys.modules[name] = mod
    for cell in nbformat.read(path, as_version=4).cells:
        if cell.cell_type == "code":
            exec(compile(cell.source, path, "exec"), mod.__dict__)
    return mod


config = import_notebook("config", "00_config.ipynb")

In [12]:
core = import_notebook("clinxai.core", "02_core_methods.ipynb")

In [13]:
import argparse
import json
import os
import sys
import time
import numpy as np
import pandas as pd
from scipy.stats import mannwhitneyu, spearmanr
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, brier_score_loss, roc_auc_score
from sklearn.model_selection import train_test_split
MODELS = ["LR", "EBM", "XGB", "MLP"]
CAT = ["gender", "admission_type", "first_careunit"]
DROP = ["stay_id", "subject_id", "anchor_year_group", "hospital_expire_flag", "era"]
N_EXPLAIN = 100
N_BG = 50

### `calib` (function)

In [14]:
def calib(y, p):
    lp = np.log(np.clip(p, 1e-6, 1 - 1e-6) / (1 - np.clip(p, 1e-6, 1 - 1e-6)))
    m = LogisticRegression(C=1e6).fit(lp.reshape(-1, 1), y)
    return float(m.coef_[0, 0]), float(m.intercept_[0])

### `boot` (function)

In [15]:
def boot(y, p, fn, n=500, seed=0):
    rng = np.random.default_rng(seed)
    v = [fn(y[i], p[i]) for i in (rng.integers(0, len(y), len(y)) for _ in range(n)) if 0 < y[i].mean() < 1]
    return np.percentile(v, [2.5, 97.5])

### `smd` (function)

In [16]:
def smd(a, b):
    a, b = a.dropna(), b.dropna()
    s = np.sqrt((a.var() + b.var()) / 2)
    return float((a.mean() - b.mean()) / s) if s > 0 else 0.0

### `tex_table` (function)

In [17]:
def tex_table(df, path, caption, label, fmt=3):
    cols = "l" * 1 + "r" * (df.shape[1] - 1)
    esc = lambda h: h if "$" in h else h.replace("_", r"\_")
    lines = [r"\begin{table*}[!htbp]\centering\scriptsize", rf"\caption{{{caption}}}\label{{{label}}}",
             r"\begin{adjustbox}{max width=\linewidth}", rf"\begin{{tabular}}{{{cols}}}", r"\toprule", " & ".join(esc(str(c)) for c in df.columns) + r" \\",
             r"\midrule"]
    for _, r in df.iterrows():
        cells = [f"{v:.{fmt}f}" if isinstance(v, (float, np.floating)) else str(v) for v in r.values]
        lines.append(" & ".join(c.replace("_", r"\_") for c in cells) + r" \\")
    lines += [r"\bottomrule", r"\end{tabular}", r"\end{adjustbox}", r"\end{table*}"]
    open(path, "w").write("\n".join(lines))

### `main` (function)

In [18]:
def main(cohort, out, seed=0):
    os.makedirs(out, exist_ok=True)
    t0 = time.time()
    df = pd.read_csv(cohort)
    feats = [c for c in df.columns if c not in DROP]
    dev, tmp = df[df.era == "2008-2016"].reset_index(drop=True), df[df.era == "2017-2022"].reset_index(drop=True)
    print(f"development era n={len(dev)}, temporal era n={len(tmp)}", flush=True)

    # ---- 1. feature-mapping consistency
    rows = []
    for c in feats:
        if c in CAT:
            continue
        rows.append(dict(feature=c, missing_dev=dev[c].isna().mean(), missing_temp=tmp[c].isna().mean(),
                         median_dev=dev[c].median(), median_temp=tmp[c].median(), smd=smd(dev[c], tmp[c])))
    fm = pd.DataFrame(rows)
    fm.to_csv(os.path.join(out, "feature_mapping.csv"), index=False)

    X, y = dev[feats], dev.hospital_expire_flag.values.astype(int)
    tr, te = train_test_split(np.arange(len(y)), test_size=0.3, stratify=y, random_state=seed)
    enc = core.Encoder().fit(X.iloc[tr], CAT)
    Ztr, Zin = enc.transform(X.iloc[tr]), enc.transform(X.iloc[te])
    Ztm, ytm = enc.transform(tmp[feats]), tmp.hospital_expire_flag.values.astype(int)
    yin = y[te]
    base = enc.baseline(Ztr)
    rng = np.random.default_rng(seed)
    Zbg = Ztr[rng.choice(len(Ztr), N_BG, replace=False)]
    xi, _ = train_test_split(np.arange(len(te)), train_size=N_EXPLAIN, stratify=yin, random_state=seed)
    xt, _ = train_test_split(np.arange(len(tmp)), train_size=N_EXPLAIN, stratify=ytm, random_state=seed)
    Zxi, Zxt = Zin[xi], Ztm[xt]

    # era-refit encoder/model inputs (temporal era, 70% for refit)
    tr2, _ = train_test_split(np.arange(len(ytm)), test_size=0.3, stratify=ytm, random_state=seed)

    perf, inst, agree, glob, transfer = [], [], [], [], []
    for kind in MODELS:
        tm = time.time()
        sub = tr if len(tr) <= 15000 else train_test_split(tr, train_size=10000, stratify=y[tr], random_state=seed)[0]
        params = core.tune(kind, enc, enc.transform(X.iloc[sub]), y[sub], seed=seed, n_iter=8)
        model = core.make_model(kind, enc, seed, params).fit(Ztr, y[tr])
        f = core.proba_fn(model)
        for era, Z, yy in (("internal", Zin, yin), ("temporal", Ztm, ytm)):
            pp = f(Z)
            lo, hi = boot(yy, pp, roc_auc_score)
            sl, ic = calib(yy, pp)
            perf.append(dict(model=kind, era=era, n=len(yy), prevalence=yy.mean(), auroc=roc_auc_score(yy, pp),
                             auroc_lo=lo, auroc_hi=hi, auprc=average_precision_score(yy, pp),
                             brier=brier_score_loss(yy, pp), calib_slope=sl, calib_intercept=ic))
        res = {}
        for era, Zx in (("internal", Zxi), ("temporal", Zxt)):
            r, _ = core.explain_all(kind, model, enc, Ztr, Zbg, Zx, seed=seed)
            res[era] = r
            names = [a for a in r if a != "Random"]
            Zp = core.perturb(Zx, enc, Ztr, 0.05, np.random.default_rng(100))
            rp, _ = core.explain_all(kind, model, enc, Ztr, Zbg, Zp, seed=seed, which=["KernelSHAP", "LIME", "native"])
            for a in names + ["Random"]:
                mo, le = core.deletion_curves(f, Zx, r[a], base)
                sp = core.stability(r[a], rp[a])[0] if a != "Random" else np.full(len(Zx), np.nan)
                ent = core.entropy(r[a])
                for i in range(len(Zx)):
                    inst.append(dict(model=kind, era=era, explainer=a, morf=mo[i], lerf=le[i], gap=mo[i] - le[i],
                                     pert_sp=sp[i], entropy=ent[i]))
            for i1, a in enumerate(names):
                for b in names[i1 + 1:]:
                    fa, sa, rc = core.pairwise_agreement(r[a], r[b])
                    agree.append(dict(model=kind, era=era, pair=f"{a} vs {b}", feat_agree=fa.mean(),
                                      sign_agree=sa.mean(), rank_corr=rc.mean()))
            for a in names:
                g = np.abs(r[a]).mean(0)
                for j, c in enumerate(enc.cols):
                    glob.append(dict(model=kind, era=era, explainer=a, feature=c, importance=g[j]))
        # ---- explanation transfer: global ranks internal vs temporal; dev model vs era-refit model
        enc2 = core.Encoder().fit(tmp[feats].iloc[tr2], CAT)  # refit uses its own preprocessing statistics
        Z2tr = enc2.transform(tmp[feats].iloc[tr2])
        m2 = core.make_model(kind, enc2, seed, params).fit(Z2tr, ytm[tr2])
        Zxt2 = enc2.transform(tmp[feats].iloc[xt])
        Zbg2 = Z2tr[np.random.default_rng(seed).choice(len(Z2tr), N_BG, replace=False)]
        r2, _ = core.explain_all(kind, m2, enc2, Z2tr, Zbg2, Zxt2, seed=seed, which=["KernelSHAP", "LIME", "native"])
        for a in r2:
            gi, gt = np.abs(res["internal"][a]).mean(0), np.abs(res["temporal"][a]).mean(0)
            g2 = np.abs(r2[a]).mean(0)
            top = lambda g: set(np.argsort(-g)[:10])
            loc_sp, loc_j = core.stability(res["temporal"][a], r2[a])
            transfer.append(dict(model=kind, explainer=a,
                                 global_sp_internal_vs_temporal=spearmanr(gi, gt).correlation,
                                 top10_overlap_internal_vs_temporal=len(top(gi) & top(gt)) / 10,
                                 global_sp_dev_vs_refit=spearmanr(gt, g2).correlation,
                                 top10_overlap_dev_vs_refit=len(top(gt) & top(g2)) / 10,
                                 local_sp_dev_vs_refit=loc_sp.mean(), local_jac_dev_vs_refit=loc_j.mean()))
        print(f"{kind} done ({time.time() - tm:.0f}s)", flush=True)

    perf, inst, agree, glob, transfer = map(pd.DataFrame, (perf, inst, agree, glob, transfer))
    for n_, d_ in dict(performance=perf, instance_metrics=inst, agreement=agree, global_importance=glob,
                       transfer=transfer).items():
        d_.to_csv(os.path.join(out, f"{n_}.csv"), index=False)

    # ---- era comparison of explanation quality (Mann-Whitney U per model x explainer)
    comp = []
    for (k, a), g in inst[inst.explainer != "Random"].groupby(["model", "explainer"]):
        gi, gt = g[g.era == "internal"], g[g.era == "temporal"]
        row = dict(model=k, explainer=a)
        for m in ["gap", "pert_sp", "entropy"]:
            row[f"{m}_internal"] = gi[m].mean()
            row[f"{m}_temporal"] = gt[m].mean()
            row[f"{m}_p"] = mannwhitneyu(gi[m], gt[m]).pvalue
        comp.append(row)
    comp = pd.DataFrame(comp)
    comp.to_csv(os.path.join(out, "era_comparison.csv"), index=False)

    json.dump(dict(runtime_s=time.time() - t0, n_dev=len(dev), n_temp=len(tmp), features=feats,
                   prev_dev=float(dev.hospital_expire_flag.mean()), prev_temp=float(tmp.hospital_expire_flag.mean())),
              open(os.path.join(out, "meta.json"), "w"), indent=1)
    write_outputs(out)
    print(f"finished in {time.time() - t0:.0f}s; outputs in {out}")

### `write_outputs` (function)

Write LaTeX tables and macros from the CSV outputs (re-runnable with --tables-only).

In [19]:
def write_outputs(out):
    """Write LaTeX tables and macros from the CSV outputs (re-runnable with --tables-only)."""
    rd = lambda n: pd.read_csv(os.path.join(out, f"{n}.csv"))
    fm, perf, comp, transfer = rd("feature_mapping"), rd("performance"), rd("era_comparison"), rd("transfer")
    meta = json.load(open(os.path.join(out, "meta.json")))
    tex_table(fm.round(3), os.path.join(out, "tab_mimic_features.tex"),
              "MIMIC-IV feature-mapping check: missingness, medians and standardised mean difference (SMD) "
              "between the development (2008--2016) and temporal (2017--2022) eras.", "tab:mimicfeat")
    # ---- LaTeX tables
    t = perf.pivot_table(index="model", columns="era", values=["auroc", "auprc", "brier", "calib_slope"]).round(3)
    nice = {"auroc": "AUROC", "auprc": "AUPRC", "brier": "Brier", "calib_slope": "Calib. slope"}
    t.columns = [f"{nice[a]} ({b})" for a, b in t.columns]
    t = t.reset_index().rename(columns={"model": "Model"})
    tex_table(t, os.path.join(out, "tab_mimic_perf.tex"),
              "MIMIC-IV discrimination and calibration: development hold-out (internal) versus 2017--2022 (temporal).",
              "tab:mimicperf")
    tc = comp[["model", "explainer", "gap_internal", "gap_temporal", "gap_p", "pert_sp_internal",
               "pert_sp_temporal", "pert_sp_p"]].round(3)
    tc.columns = ["Model", "Explainer", "Gap (int.)", "Gap (temp.)", "p", "Stab. (int.)", "Stab. (temp.)", "p "]
    tex_table(tc, os.path.join(out, "tab_mimic_expl.tex"),
              "MIMIC-IV explanation quality by era. Gap: MoRF$-$LeRF deletion gap; Stab.: Spearman stability under "
              "5\\% input noise; p: two-sided Mann--Whitney U.", "tab:mimicexpl")
    tt = transfer.round(3)
    tt.columns = ["Model", "Explainer", "Global $\\rho$ int.-temp.", "Top-10 int.-temp.", "Global $\\rho$ dev-refit",
                  "Top-10 dev-refit", "Local $\\rho$ dev-refit", "Local Jacc. dev-refit"]
    tex_table(tt, os.path.join(out, "tab_mimic_transfer.tex"),
              "MIMIC-IV explanation transfer across eras.", "tab:mimictransfer")

    # ---- macros for the manuscript text
    def mac(name, v, d=3):
        return f"\\newcommand{{\\{name}}}{{{v:.{d}f}}}" if isinstance(v, (float, np.floating)) else \
               f"\\newcommand{{\\{name}}}{{{v}}}"
    M = [mac("mimicNdev", f"{meta['n_dev']:,}"), mac("mimicNtemp", f"{meta['n_temp']:,}"),
         mac("mimicPrevDev", 100 * meta["prev_dev"], 1), mac("mimicPrevTemp", 100 * meta["prev_temp"], 1),
         mac("mimicNfeat", str(len(meta["features"]))),
         mac("mimicMaxSMD", fm.smd.abs().max()), mac("mimicMaxSMDfeat", fm.loc[fm.smd.abs().idxmax(), "feature"].replace("_", r"\_")),
         mac("mimicNsmdTen", str(int((fm.smd.abs() > 0.1).sum())))]
    for k in MODELS:
        for era in ("internal", "temporal"):
            r = perf[(perf.model == k) & (perf.era == era)].iloc[0]
            M.append(mac(f"mimicAUC{k}{era}", r.auroc))
    M.append(mac("mimicTransferMedianGlobal", transfer.global_sp_internal_vs_temporal.median()))
    M.append(mac("mimicTransferMedianRefit", transfer.global_sp_dev_vs_refit.median()))
    M.append(mac("mimicTransferMedianLocal", transfer.local_sp_dev_vs_refit.median()))
    M.append(mac("mimicNsigGap", str(int((comp.gap_p < 0.05).sum()))))
    M.append(mac("mimicNsigStab", str(int((comp.pert_sp_p < 0.05).sum()))))
    M.append(mac("mimicNcomp", str(len(comp))))
    open(os.path.join(out, "mimic_macros.tex"), "w").write("\n".join(M) + "\n\\newcommand{\\mimicavailable}{1}\n")

## Run
Needs the cohort written by `06_mimic_extract_cohort`. Takes 1-3 hours. Writes CSVs, LaTeX tables and `mimic_macros.tex` to `OUTPUT_DIR/results_mimic`; copy that folder next to the `paper/` folder (as `results_mimic/`) and recompile the manuscript. Set `TABLES_ONLY = True` to rebuild only the LaTeX files from existing CSVs.

In [20]:
TABLES_ONLY = False
if TABLES_ONLY:
    write_outputs(config.MIMIC_RESULTS_DIR)
else:
    main(config.MIMIC_COHORT, config.MIMIC_RESULTS_DIR)

development era n=34679, temporal era n=16988
LR done (62s)
EBM done (87s)
XGB done (80s)
MLP done (123s)
finished in 352s; outputs in D:\New_Research_2027\Clinical_XAI_Faithfulness\outputs\results_mimic
